In [22]:
# PRT564 Data Analytics and Visualisation
# Assessment 2: Regression Pipeline
# Dataset : Energy Rating Data for Household Appliances (Air Conditioners)
# Source  : DCCEEW via data.gov.au
# External: Clothes Dryer Energy Ratings CSV  heterogeneous integration
# Source  : data.gov.auJoined on Brand (uppercase-normalised) via brand-level aggregate features
# Models  : OLS Linear Regression (Model 1) + Random Forest (Model 2)



# =========================== STEP 1 Import all required libraries ======================

try:
    import pandas as pd
    import numpy as np
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import seaborn as sns

    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.impute import SimpleImputer

    from scipy import stats

    import json, warnings
    warnings.filterwarnings('ignore')

    print("\n  All libraries loaded successfully.\n")

except ImportError as e:
    print(f"\n  IMPORT ERROR: {e}")
    raise SystemExit(1)


# =========================== STEP 2 Load the dataset ======================

filepath_ac = '/content/drive/MyDrive/data_analytics/ac_2026_03_17.csv'   # Air conditioner ratings
filepath_cd = '/content/drive/MyDrive/data_analytics/cd_2026_04_21.csv'   # Clothes dryer ratings (heterogeneous source 2)

try:
    df = pd.read_csv(filepath_ac, low_memory=False)

    n_total = len(df)
    print(f"  Dataset loaded successfully.")
    print(f"     Total rows    : {n_total}")
    print(f"     Total columns : {len(df.columns)}")
    # print(f"     Columns       : {list(df.columns)}")
    missing_summary = df.isnull().sum()
    missing_cols    = missing_summary[missing_summary > 0]
    if len(missing_cols) > 0:
        print(f"\n  Missing values detected in {len(missing_cols)} columns.")
    else:
        print("  No missing values detected in raw data")
    print()

except FileNotFoundError as e:
    print(f"\n  FILE NOT FOUND: {e}")
    print("     Make sure 'ac_2026_03_17.csv' is in the working directory.")
    raise SystemExit(1)
except pd.errors.ParserError as e:
    print(f"\n  CSV PARSE ERROR: {e}")
    print("     The CSV file may be corrupted or use an unexpected delimiter.")
    raise SystemExit(1)
except Exception as e:
    print(f"\n  UNEXPECTED ERROR while loading data: {e}")
    raise


# =========================== STEP 3 Integrate external data ======================

print("=" * 65)
print("STEP 3: Heterogeneous data integration")
print("=" * 65)


# 3 Clothes Dryer Energy Ratings CSV
try:
    df_cd = pd.read_csv(filepath_cd, low_memory=False)
    # print(f"  Dryer dataset loaded: {len(df_cd)} rows, {len(df_cd.columns)} columns")

    # Normalise brand name in dryer dataset
    df_cd['Brand_upper'] = df_cd['Brand'].str.upper().str.strip()

    # Aggregate to brand level — one row per brand
    cd_brand_agg = (
        df_cd
        .groupby('Brand_upper', as_index=False)
        .agg(
            CD_Avg_Energy_kWh  =('Labelled energy consumption (kWh/year)', 'mean'),
            CD_Avg_Star        =('New Star', 'mean'),
            CD_Avg_Cap_kg      =('Cap',      'mean'),
            CD_Model_Count     =('Brand_upper', 'count'),
        )
    )
    cd_brand_agg[['CD_Avg_Energy_kWh','CD_Avg_Star','CD_Avg_Cap_kg']] = \
        cd_brand_agg[['CD_Avg_Energy_kWh','CD_Avg_Star','CD_Avg_Cap_kg']].round(3)

    # print(f"  Brand-level aggregation produced {len(cd_brand_agg)} unique brands")
    # print(f"  Sample brand aggregates (top 5 by model count):")
    # for _, row in cd_brand_agg.nlargest(5, 'CD_Model_Count').iterrows():
    #     print(f"      {row['Brand_upper']:<25} "
    #           f"energy={row['CD_Avg_Energy_kWh']:.0f} kWh/yr  "
    #           f"stars={row['CD_Avg_Star']:.1f}  "
    #           f"cap={row['CD_Avg_Cap_kg']:.1f} kg  "
    #           f"({int(row['CD_Model_Count'])} models)")

    # Normalise brand in AC dataframe and left-join
    df['Brand_upper'] = df['Brand'].str.upper().str.strip()
    rows_before_cd = len(df)
    df = pd.merge(df, cd_brand_agg[['Brand_upper','CD_Avg_Energy_kWh',
                                     'CD_Avg_Star','CD_Avg_Cap_kg']],
                  on='Brand_upper', how='left')

    if len(df) != rows_before_cd:
        raise ValueError(
            f"Row count changed after dryer merge: {rows_before_cd} {len(df)}. "
            "Check for duplicate Brand_upper keys in cd_brand_agg."
        )

    matched_cd  = df['CD_Avg_Star'].notna().sum()
    unmatched_cd = df['CD_Avg_Star'].isna().sum()
    print(f"\n  Dryer brand merge results:")
    print(f"      AC rows matched to dryer brand data : {matched_cd}  ({matched_cd/len(df)*100:.1f}%)")
    print(f"      AC rows unmatched (no dryer data)   : {unmatched_cd} ({unmatched_cd/len(df)*100:.1f}%) will be imputed")
    print(f"      Row count preserved                 : {len(df)}")
    print(f"  New columns added: CD_Avg_Energy_kWh, CD_Avg_Star, CD_Avg_Cap_kg\n")

except FileNotFoundError:
    print(f"  WARNING: '{filepath_cd}' not found — skipping dryer integration.")
    print("      CD features will be absent; Step 6 will exclude them automatically.")
    df['CD_Avg_Energy_kWh'] = np.nan
    df['CD_Avg_Star']       = np.nan
    df['CD_Avg_Cap_kg']     = np.nan
except KeyError as e:
    print(f"\n  KEY ERROR during 3B merge: {e}")
    raise
except ValueError as e:
    print(f"\n  VALUE ERROR during 3B merge: {e}")
    raise
except Exception as e:
    print(f"\n  UNEXPECTED ERROR during 3B integration: {e}")
    raise


# =========================== STEP 4 Preprocessing (filter, clean, encode) ======================

print("=" * 65)
print("STEP 4: Filtering, encoding, and feature engineering")
print("        (Pre-split preprocessing steps)")
print("=" * 65)

try:
    # 4a. Keep only currently approved registrations
    n_before_filter = len(df)
    df = df[df['SubmitStatus'] == 'Approved'].copy()
    n_after_status  = len(df)
    print(f"  Filtered to Approved status: {n_before_filter} {n_after_status} records"
          f" ({n_before_filter - n_after_status} removed)")

    # Rename raw CSV columns to clean internal names used throughout the script
    df = df.rename(columns={
        'Rated AEER':                  'Rated_AEER',
        'Rated Total Cool Capacity W': 'Rated_Cool_Cap_W',
        'Rated cooling power input kW':'Rated_Cool_Pwr_kW',
        'indoor_sound_level':          'indoor_sound_dB',
        'outdoor_sound_level':         'outdoor_sound_dB',
    })

    # 4b. Drop rows where the target variable (AEER) is completely missing
    n_before_dropna = len(df)
    df = df.dropna(subset=['Rated_AEER'])
    n_approved = len(df)
    dropped_target = n_before_dropna - n_approved
    if dropped_target > 0:
        print(f"  Dropped {dropped_target} rows where Rated_AEER was missing")
    else:
        print(f"  No rows dropped for missing AEER")

    # Check Create Is_Inverter binary flag from the 'Invert' column
    df['Is_Inverter'] = (df['Invert'].astype(str).str.strip().str.lower() == 'yes').astype(int)
    # print(f"  Is_Inverter flag created from 'Invert' column:")
    # print(f"      Inverter (1): {df['Is_Inverter'].sum()}  |  Fixed speed (0): {(df['Is_Inverter']==0).sum()}")

    # Convert sound-level columns from mixed string/numeric to float
    df['indoor_sound_dB']  = pd.to_numeric(df['indoor_sound_dB'],  errors='coerce')
    df['outdoor_sound_dB'] = pd.to_numeric(df['outdoor_sound_dB'], errors='coerce')
    # print(f"  Sound-level columns converted to numeric "
    #       f"(indoor missing: {df['indoor_sound_dB'].isna().sum()}, "
    #       f"outdoor missing: {df['outdoor_sound_dB'].isna().sum()})")

    # 4c. One-hot encoding — drop_first=True prevents the dummy variable trap
    cat_cols_before = ['Configuration1', 'Refrigerant', 'Phase', 'Climate_Zone']
    missing_cats    = [c for c in cat_cols_before if c not in df.columns]
    if missing_cats:
        # print(f"  These categorical columns were not found and skipped: {missing_cats}")
        cat_cols_before = [c for c in cat_cols_before if c in df.columns]

    df = pd.get_dummies(df, columns=cat_cols_before, drop_first=True)
    new_dummy_cols = [c for c in df.columns if any(
        c.startswith(p) for p in ['Configuration1_','Refrigerant_','Phase_','Climate_Zone_'])]
    # print(f"  One-hot encoding applied to: {cat_cols_before}")
    # print(f"      Dummy columns created: {new_dummy_cols}")

    # 4d. Log-transform cooling capacity (strongly right-skewed)
    if 'Rated_Cool_Cap_W' not in df.columns:
        raise KeyError("'Rated_Cool_Cap_W' column not found — required for log transform.")

    # Convert to numeric first (may contain '-' strings) then check for negatives
    df['Rated_Cool_Cap_W'] = pd.to_numeric(df['Rated_Cool_Cap_W'], errors='coerce')
    if (df['Rated_Cool_Cap_W'].dropna() < 0).any():
        raise ValueError("Negative values detected in 'Rated_Cool_Cap_W' — log transform invalid.")

    df['Log_Cool_Cap'] = np.log1p(df['Rated_Cool_Cap_W'])
    # print(f"  Log-transform applied to Rated_Cool_Cap_W Log_Cool_Cap")
    # print(f"      Original range: {df['Rated_Cool_Cap_W'].min():.0f} – {df['Rated_Cool_Cap_W'].max():.0f} W")
    # print(f"      Log range     : {df['Log_Cool_Cap'].min():.2f} – {df['Log_Cool_Cap'].max():.2f}")

    # Ensure power column is numeric before arithmetic
    df['Rated_Cool_Pwr_kW'] = pd.to_numeric(df['Rated_Cool_Pwr_kW'], errors='coerce')

    # 4e. Derive EER as cross-check on published values
    df['EER_Derived'] = df['Rated_Cool_Cap_W'] / (df['Rated_Cool_Pwr_kW'] * 1000)
    # print(f"  EER_Derived computed as cross-check (Cap / Power)")

    print(f"\n  Pre-split preprocessing complete. Records remaining: {n_approved}\n")

except KeyError as e:
    print(f"\n  KEY ERROR in preprocessing: {e}")
    raise
except ValueError as e:
    print(f"\n  VALUE ERROR in preprocessing: {e}")
    raise
except Exception as e:
    print(f"\n  UNEXPECTED ERROR in preprocessing: {e}")
    raise


# STEP 5 — Exploratory Data Analysis (EDA) and visualisations

print("=" * 65)
print("STEP 5: Exploratory Data Analysis — generating visualisations")
print("=" * 65)

TEAL  = '#028090'
NAVY  = '#21295C'
MINT  = '#02C39A'
CORAL = '#F96167'
plt.style.use('seaborn-v0_8-whitegrid')

# Plot 1: AEER distribution + log-capacity distribution
try:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df['Rated_AEER'].dropna(), bins=35,
                 color=TEAL, edgecolor='white', linewidth=0.5)
    axes[0].set_title('Distribution of Rated AEER', fontsize=13,
                      fontweight='bold', color=NAVY)
    axes[0].set_xlabel('Annual Energy Efficiency Ratio (AEER)', color=NAVY)
    axes[0].axvline(df['Rated_AEER'].mean(), color=CORAL, linestyle='--',
                    linewidth=2, label=f"Mean={df['Rated_AEER'].mean():.2f}")
    axes[0].legend()

    axes[1].hist(df['Log_Cool_Cap'].dropna(), bins=35,
                 color=MINT, edgecolor='white', linewidth=0.5)
    axes[1].set_title('Log-Transformed Cooling Capacity', fontsize=13,
                      fontweight='bold', color=NAVY)
    axes[1].set_xlabel('log(Rated Cooling Capacity W)', color=NAVY)

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plot1_distributions.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plot1_distributions.png saved")

except Exception as e:
    print(f"  ERROR generating Plot 1: {e}")
    plt.close('all')

# Plot 2: AEER by configuration and by inverter type
try:
    # print(" Generating Plot 2: AEER by configuration and inverter type...")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    config_col = [c for c in df.columns if 'Configuration1' in c]
    if config_col:
        df['Config_label'] = np.where(
            df[config_col[0]] == True, 'Non Ducted', 'Ducted/Packaged')
        df.boxplot(column='Rated_AEER', by='Config_label', ax=axes[0],
                   boxprops=dict(color=NAVY),
                   medianprops=dict(color=CORAL, linewidth=2))
        axes[0].set_title('AEER by Configuration', fontsize=12,
                          fontweight='bold', color=NAVY)
        plt.sca(axes[0])
        plt.title('AEER by Configuration', fontsize=12, fontweight='bold')
    else:
        axes[0].text(0.5, 0.5, 'Configuration column\nnot available',
                     ha='center', va='center', transform=axes[0].transAxes)
        print("  No Configuration1 dummy column found — skipping first boxplot")

    if 'Is_Inverter' not in df.columns:
        raise KeyError("'Is_Inverter' column missing — cannot generate inverter boxplot.")
    df['Inverter_label'] = df['Is_Inverter'].map({1: 'Inverter', 0: 'Fixed Speed'})
    df.boxplot(column='Rated_AEER', by='Inverter_label', ax=axes[1],
               boxprops=dict(color=NAVY),
               medianprops=dict(color=MINT, linewidth=2))
    axes[1].set_title('AEER by Technology', fontsize=12,
                      fontweight='bold', color=NAVY)
    plt.sca(axes[1])
    plt.title('AEER by Compressor Type', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plot2_boxplots.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plot2_boxplots.png saved")

except KeyError as e:
    print(f"  KEY ERROR generating Plot 2: {e}")
    plt.close('all')
except Exception as e:
    print(f"  ERROR generating Plot 2: {e}")
    plt.close('all')

# Plot 3: Pearson correlation heatmap (key variables)
try:
    # print(" Generating Plot 3: Pearson correlation heatmap (key variables)...")
    num_features = ['Rated_AEER','Log_Cool_Cap','Rated_Cool_Pwr_kW',
                    'EER_Derived','Is_Inverter',
                    'indoor_sound_dB','outdoor_sound_dB','Climate_EER_Adj',
                    'CD_Avg_Energy_kWh','CD_Avg_Star','CD_Avg_Cap_kg']
    num_features = [c for c in num_features if c in df.columns]

    if len(num_features) < 2:
        raise ValueError("Fewer than 2 numeric features available — cannot compute correlation.")

    corr = df[num_features].corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, ax=ax, cbar_kws={'shrink': 0.8},
                linewidths=0.5, annot_kws={'size': 9})
    ax.set_title('Pearson Correlation Matrix — Key Variables',
                 fontsize=13, fontweight='bold', color=NAVY, pad=15)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plot3_correlation.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plot3_correlation.png saved")

except ValueError as e:
    print(f"  VALUE ERROR generating Plot 3: {e}")
    plt.close('all')
except Exception as e:
    print(f"  ERROR generating Plot 3: {e}")
    plt.close('all')

# Plot 4: Scatter — log-capacity vs AEER coloured by inverter
try:
    # print(" Generating Plot 4: Cooling capacity vs AEER scatter plot...")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    sc = ax.scatter(df['Log_Cool_Cap'], df['Rated_AEER'],
                    c=df['Is_Inverter'], cmap='coolwarm', alpha=0.45, s=18)
    ax.set_xlabel('log(Cooling Capacity W)', color=NAVY, fontsize=11)
    ax.set_ylabel('Rated AEER', color=NAVY, fontsize=11)
    ax.set_title('Cooling Capacity vs AEER (colour = Inverter Technology)',
                 fontsize=12, fontweight='bold', color=NAVY)
    cbar = plt.colorbar(sc, ax=ax, ticks=[0, 1])
    cbar.set_ticklabels(['Fixed Speed', 'Inverter'])
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plot4_scatter.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plot4_scatter.png saved")

except Exception as e:
    print(f"  ERROR generating Plot 4: {e}")
    plt.close('all')

# Plot A: Power input by brand (boxplot)
try:
    # print(" Generating Plot A: c_power_inp_rated by brand...")
    if 'Brand' not in df.columns:
        raise KeyError("'Brand' column not found in dataframe.")

    top_brands = df['Brand'].value_counts().head(10).index.tolist()
    df_top     = df[df['Brand'].isin(top_brands)]
    brand_data = [
        df_top.loc[df_top['Brand'] == b, 'Rated_Cool_Pwr_kW'].dropna().values
        for b in top_brands
    ]
    valid = [(b, d) for b, d in zip(top_brands, brand_data) if len(d) > 0]
    if not valid:
        raise ValueError("No valid brand data available for boxplot.")
    top_brands_valid, brand_data_valid = zip(*valid)

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.boxplot(
        brand_data_valid, labels=top_brands_valid, patch_artist=True,
        showfliers=True,
        flierprops=dict(marker='o', markerfacecolor='none',
                        markeredgecolor='#21295C', markersize=4, alpha=0.5),
        medianprops=dict(color='white', linewidth=2),
        boxprops=dict(facecolor='#3A5F8A', color='#21295C'),
        whiskerprops=dict(color='#21295C', linewidth=1),
        capprops=dict(color='#21295C', linewidth=1.5),
    )
    ax.set_title('c_power_inp_rated by brand', fontsize=14, pad=12)
    ax.set_xlabel('brand', fontsize=12, labelpad=10)
    ax.set_ylabel('c_power_inp_rated', fontsize=12)
    ax.tick_params(axis='x', rotation=45, labelsize=10)
    ax.set_facecolor('#F8F8F8')
    fig.patch.set_facecolor('white')
    ax.grid(axis='y', linestyle='-', linewidth=0.5, color='#CCCCCC', alpha=0.7)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plotA_brand_boxplot.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plotA_brand_boxplot.png saved")

except KeyError as e:
    print(f"  KEY ERROR generating Plot A: {e}")
    plt.close('all')
except ValueError as e:
    print(f"  VALUE ERROR generating Plot A: {e}")
    plt.close('all')
except Exception as e:
    print(f"  ERROR generating Plot A: {e}")
    plt.close('all')

# Plot B: Power input distribution with KDE
try:
    # print(" Generating Plot B: Distribution of c_power_inp_rated with KDE...")
    pwr_vals = df['Rated_Cool_Pwr_kW'].dropna()
    if len(pwr_vals) < 10:
        raise ValueError("Insufficient data points for KDE estimation.")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(pwr_vals, bins=60, color='#5B8DB8', edgecolor='white',
            linewidth=0.3, density=False, alpha=0.85)

    kde       = stats.gaussian_kde(pwr_vals, bw_method='scott')
    kde_xs    = np.linspace(pwr_vals.min(), pwr_vals.max(), 400)
    bin_width = (pwr_vals.max() - pwr_vals.min()) / 60
    ax.plot(kde_xs, kde(kde_xs) * len(pwr_vals) * bin_width,
            color='#21295C', linewidth=2)

    ax.set_title('Distribution of c_power_inp_rated', fontsize=14, pad=12)
    ax.set_xlabel('c_power_inp_rated', fontsize=12, labelpad=10)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_xlim(left=0)
    ax.axvline(pwr_vals.mean(), color=CORAL, linestyle='--', linewidth=1.5,
               label=f'Mean = {pwr_vals.mean():.2f} kW')
    ax.axvline(pwr_vals.median(), color=MINT, linestyle=':', linewidth=1.5,
               label=f'Median = {pwr_vals.median():.2f} kW')
    ax.legend(fontsize=10)
    ax.grid(linestyle='-', linewidth=0.5, color='#DDDDDD', alpha=0.8)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plotB_power_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plotB_power_distribution.png saved")

except ValueError as e:
    print(f"  VALUE ERROR generating Plot B: {e}")
    plt.close('all')
except Exception as e:
    print(f"  ERROR generating Plot B: {e}")
    plt.close('all')

# Plot C: Full correlation heatmap — RdBu_r
try:
    # print(" Generating Plot C: Full correlation heatmap (RdBu_r)...")
    numeric_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    drop_cols_viz = ['Is_Inverter','Log_Cool_Cap','EER_Derived',
                     'Climate_EER_Adj','c_star_mixed']
    heatmap_cols  = [c for c in numeric_cols if c not in drop_cols_viz]

    if len(heatmap_cols) < 2:
        raise ValueError("Fewer than 2 numeric columns available for full heatmap.")

    corr_full = df[heatmap_cols].corr()
    n_c       = len(heatmap_cols)
    fsz       = max(14, n_c * 0.65)
    fig, ax   = plt.subplots(figsize=(fsz, fsz * 0.85))
    sns.heatmap(corr_full, ax=ax, cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, annot=False, linewidths=0.3,
                linecolor='#EEEEEE', square=True,
                cbar_kws={'label':'Correlation','shrink':0.6,'pad':0.02})
    ax.set_title('Correlation Heatmap', fontsize=15, fontweight='normal', pad=16)
    ax.tick_params(axis='x', rotation=90, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=9)
    fig.patch.set_facecolor('white')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plotC_full_correlation.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plotC_full_correlation.png saved")

except ValueError as e:
    print(f"  VALUE ERROR generating Plot C: {e}")
    plt.close('all')
except Exception as e:
    print(f"  ERROR generating Plot C: {e}")
    plt.close('all')

# Plot D: Dryer brand features vs AC AEER (heterogeneous integration visualisation)
try:
    # print("  Generating Plot D: Dryer brand features vs AC AEER...")
    cd_plot_cols = [c for c in ['CD_Avg_Energy_kWh','CD_Avg_Star','CD_Avg_Cap_kg']
                    if c in df.columns and df[c].notna().sum() > 10]
    if not cd_plot_cols:
        raise ValueError("No dryer CD columns available for Plot D.")

    fig, axes = plt.subplots(1, len(cd_plot_cols), figsize=(6 * len(cd_plot_cols), 4.5))
    if len(cd_plot_cols) == 1:
        axes = [axes]

    labels = {
        'CD_Avg_Energy_kWh': "Brand Avg Dryer Energy (kWh/yr)",
        'CD_Avg_Star':        "Brand Avg Dryer Star Rating",
        'CD_Avg_Cap_kg':      "Brand Avg Dryer Capacity (kg)",
    }
    for ax, col in zip(axes, cd_plot_cols):
        subset = df[['Rated_AEER', col]].dropna()
        ax.scatter(subset[col], subset['Rated_AEER'],
                   alpha=0.35, s=14, color=TEAL, edgecolors='none')
        # Add linear trend line
        m, b, r, p, _ = stats.linregress(subset[col], subset['Rated_AEER'])
        xs = np.linspace(subset[col].min(), subset[col].max(), 200)
        ax.plot(xs, m * xs + b, color=CORAL, linewidth=2,
                label=f'r={r:.2f}  p={p:.3f}')
        ax.set_xlabel(labels.get(col, col), color=NAVY, fontsize=10)
        ax.set_ylabel('Rated AEER', color=NAVY, fontsize=10)
        ax.set_title(f'{labels.get(col, col)}\nvs AC AEER',
                     fontsize=11, fontweight='bold', color=NAVY)
        ax.legend(fontsize=9)

    fig.suptitle('Heterogeneous Integration: Dryer Brand Features vs AC Efficiency',
                 fontsize=12, fontweight='bold', color=NAVY, y=1.02)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plotD_dryer_vs_aeer.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plotD_dryer_vs_aeer.png saved")

except ValueError as e:
    print(f"  VALUE ERROR generating Plot D: {e}")
    plt.close('all')
except Exception as e:
    print(f"  ERROR generating Plot D: {e}")
    plt.close('all')

print(f"\n  EDA complete — all visualisations generated.\n")


# STEP 6 — Feature selection and train/test split

print("=" * 65)
print("STEP 6: Feature selection and train/test split (80:20)")
print("=" * 65)

try:
    # Core continuous features
    feature_cols = ['Log_Cool_Cap','Rated_Cool_Pwr_kW','Is_Inverter',
                    'indoor_sound_dB','outdoor_sound_dB']

    # Clothes dryer brand-level features from heterogeneous integration (3B)
    cd_features = ['CD_Avg_Energy_kWh','CD_Avg_Star','CD_Avg_Cap_kg']
    cd_features  = [c for c in cd_features if c in df.columns]
    feature_cols += cd_features
    if cd_features:
        print(f"  Dryer brand features included: {cd_features}")

    # Add one-hot dummy columns created in Step 4
    bool_dummies = [c for c in df.columns
                    if df[c].dtype in [bool, 'uint8', 'int32']]
    feature_cols += [c for c in bool_dummies
                     if c.startswith('Config') or c.startswith('Phase')
                     or c.startswith('Refrig') or c.startswith('Climate')]

    # Keep only columns that exist in the dataframe
    feature_cols = [c for c in feature_cols if c in df.columns]
    feature_cols = list(dict.fromkeys(feature_cols))  # remove duplicates

    if len(feature_cols) == 0:
        raise ValueError("No feature columns found — check preprocessing steps.")
    if 'Rated_AEER' not in df.columns:
        raise KeyError("Target variable 'Rated_AEER' not found in dataframe.")

    X = df[feature_cols].astype(float)
    y = df['Rated_AEER']

    print(f"  Features selected ({len(feature_cols)} total):")
    for fc in feature_cols:
        missing_pct = X[fc].isna().mean() * 100
        # print(f"      {fc:<40} missing: {missing_pct:.1f}%")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42)

    print(f"\n  Train/test split completed:")
    print(f"      Training set : {X_train.shape[0]} records ({X_train.shape[0]/len(X)*100:.0f}%)")
    print(f"      Test set     : {X_test.shape[0]}  records ({X_test.shape[0]/len(X)*100:.0f}%)")
    print(f"      Target (AEER): mean={y_train.mean():.3f}  std={y_train.std():.3f}\n")

except KeyError as e:
    print(f"\n  KEY ERROR in feature selection: {e}")
    raise
except ValueError as e:
    print(f"\n  VALUE ERROR in feature selection: {e}")
    raise
except Exception as e:
    print(f"\n  UNEXPECTED ERROR in feature selection: {e}")
    raise


# STEP 7 — Imputation and scaling (Post-split to prevent data leakage)

print("=" * 65)
print("STEP 7: Post-split imputation and StandardScaler normalisation")
print("        (Fit on TRAIN only. prevents data leakage)")
print("=" * 65)

try:
    cols_to_impute = [c for c in ['indoor_sound_dB','outdoor_sound_dB',
                                   'Rated_Cool_Pwr_kW','Log_Cool_Cap',
                                   'CD_Avg_Energy_kWh','CD_Avg_Star','CD_Avg_Cap_kg']
                      if c in X_train.columns]

    if not cols_to_impute:
        print("  No columns required imputation — skipping imputer step")
    else:
        imputer = SimpleImputer(strategy='median')
        X_train[cols_to_impute] = imputer.fit_transform(X_train[cols_to_impute])
        X_test[cols_to_impute]  = imputer.transform(X_test[cols_to_impute])
        # print(f"  Median imputation applied to: {cols_to_impute}")
        # print(f"      Learned medians: { {c: round(m, 3) for c, m in zip(cols_to_impute, imputer.statistics_)} }")

    # Verify no missing values remain before scaling
    remaining_missing = X_train.isnull().sum().sum()
    if remaining_missing > 0:
        raise ValueError(
            f"Still {remaining_missing} missing values in X_train after imputation. "
            "Check feature columns for unexpected NaNs."
        )

    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    print(f"  StandardScaler applied — all features now: mean≈0, std≈1")
    print(f"      Feature means (train, post-scale): "
          f"{np.round(X_train_sc.mean(axis=0)[:3], 4)} ...")
    print(f"  Imputation and scaling complete.\n")

except ValueError as e:
    print(f"\n  VALUE ERROR in imputation/scaling: {e}")
    raise
except Exception as e:
    print(f"\n  UNEXPECTED ERROR in imputation/scaling: {e}")
    raise


# STEP 8 — Model assumption checks

print("=" * 65)
print("STEP 8: Checking model assumptions (VIF + Shapiro-Wilk)")
print("=" * 65)

# --- VIF check for multicollinearity ---
try:
    print(" Running VIF (Variance Inflation Factor) check...")

    def compute_vif(X_df):
        """
        Compute VIF for each feature in X_df.
        VIF = 1/(1-R²) where R² is from regressing each column on all others.
        VIF < 5   acceptable
        VIF 5–10  moderate (monitor)
        VIF > 10  severe multicollinearity (consider removing)
        """
        vif = {}
        for col in X_df.columns:
            y_v = X_df[col].values
            X_v = X_df.drop(columns=[col]).values
            if X_v.shape[1] == 0:
                vif[col] = 1.0
                continue
            r2 = LinearRegression().fit(X_v, y_v).score(X_v, y_v)
            vif[col] = round(1 / (1 - r2) if r2 < 1.0 else float('inf'), 3)
        return vif

    vif_scores   = compute_vif(X_train)
    max_vif_feat = max(vif_scores, key=vif_scores.get)
    max_vif_val  = vif_scores[max_vif_feat]

    # print(f"  VIF scores computed for all {len(vif_scores)} features:")
    for feat, v in sorted(vif_scores.items(), key=lambda x: -x[1]):
        flag = "OK" if v < 5 else (" Moderate" if v < 10 else "HIGH")
        # print(f"  {feat:<40} VIF = {v:>7.3f}  {flag}")

    high_vif = [f for f, v in vif_scores.items() if v >= 10]
    if high_vif:
        # print(f"\n  WARNING: High VIF detected for: {high_vif}")
        print(" removing or combining these features before interpreting coefficients.")
    else:
        print(f"\n  All VIF scores acceptable (max={max_vif_val} for '{max_vif_feat}').")

except Exception as e:
    print(f"\n  ERROR during VIF check: {e}")
    max_vif_feat = "Unknown"
    max_vif_val  = -1.0

#  Shapiro-Wilk normality test on AEER
try:
    # print("\n Running Shapiro-Wilk normality test on AEER (response variable)...")
    sample_size   = min(500, len(y_train))
    sample_aeer   = y_train.sample(sample_size, random_state=42)
    sw_stat, sw_p = stats.shapiro(sample_aeer)

    print(f"  Shapiro-Wilk test (n={sample_size} sample):")
    print(f"      W statistic = {sw_stat:.4f}")
    print(f"      p-value     = {sw_p:.4f}")

    if sw_p > 0.05:
        print(" Fail to reject H₀: AEER is approximately normally distributed (α=0.05)")
    else:
        print(" Reject H₀: AEER deviates from normality (α=0.05)")
        print(" Note: OLS is robust to mild non-normality at large n (Central Limit Theorem).")
        print(f"  With n={n_approved}, OLS coefficient estimates remain valid.")

    print(f"\n  Assumption checks complete.\n")

except Exception as e:
    print(f"\n  ERROR during Shapiro-Wilk test: {e}")
    sw_stat, sw_p = -1.0, -1.0


# STEP 9 — Train and evaluate Model 1 (OLS Linear Regression)

print("=" * 65)
print("STEP 9: Training Model 1 OLS Multiple Linear Regression")
print("=" * 65)

try:
    # print(" Fitting LinearRegression(fit_intercept=True) on training set...")
    lr1 = LinearRegression(fit_intercept=True)
    lr1.fit(X_train_sc, y_train)
    y_pred1 = lr1.predict(X_test_sc)

    mae1     = mean_absolute_error(y_test, y_pred1)
    mse1     = mean_squared_error(y_test, y_pred1)
    rmse1    = np.sqrt(mse1)
    r2_1     = r2_score(y_test, y_pred1)
    n_test   = len(y_test)
    p1       = X_test_sc.shape[1]
    adj_r2_1 = 1 - (1 - r2_1) * (n_test - 1) / (n_test - p1 - 1)
    nrmse1   = rmse1 / (y_test.max() - y_test.min())
    res1     = y_test.values - y_pred1

    print(f"  Model 1 (OLS) training complete")
    print(f"  Intercept (β₀)  : {lr1.intercept_:.6f}")
    print(f"  No. coefficients: {len(lr1.coef_)}")
    print(f"\n  Model 1 performance on test set ({n_test} records):")
    print(f"  MAE      = {mae1:.4f}  (avg prediction error in AEER units)")
    print(f"  RMSE     = {rmse1:.4f}  (penalises large errors more than MAE)")
    print(f"  NRMSE    = {nrmse1:.4f}  (scale-free; 0=perfect, 1=baseline)")
    print(f"  R²       = {r2_1:.4f}  ({r2_1*100:.1f}% variance explained)")
    print(f"  Adj. R²  = {adj_r2_1:.4f}  (penalises extra predictors)")

    # Matrix inverse cross-check
    # print("\n  Cross-checking with matrix inverse formula: β = (XᵀX)⁻¹Xᵀy ...")
    try:
        X_mat    = np.column_stack([np.ones(len(X_train_sc)), X_train_sc])
        beta_inv = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ y_train.values
        intercept_diff = abs(beta_inv[0] - lr1.intercept_)
        max_coef_diff  = np.max(np.abs(beta_inv[1:] - lr1.coef_))
        print(f"  Matrix inverse intercept  : {beta_inv[0]:.6f}")
        print(f"      sklearn intercept         : {lr1.intercept_:.6f}")
        print(f"      Intercept difference      : {intercept_diff:.2e}")
        print(f"      Max coefficient difference: {max_coef_diff:.2e}")
        if intercept_diff < 1e-4:
            print("  Matrix inverse confirms sklearn OLS to high precision.")
        else:
            print("  Non-trivial difference — check for numerical instability.")
    except np.linalg.LinAlgError as e:
        # print(f"  Matrix inverse failed (singular matrix): {e}")
        beta_inv = np.array([lr1.intercept_] + list(lr1.coef_))


except Exception as e:
    print(f"\n  ERROR during Model 1 training/evaluation: {e}")
    raise


# STEP 10 — Train and evaluate Model 2 (Random Forest Regressor)

print("=" * 65)
print("STEP 10: Training Model 2 — Random Forest Regressor")
print("=" * 65)

try:
    # print(" Fitting RandomForestRegressor(n_estimators=100) on training set...")
    # print("  (This may take a few seconds — building 100 decision trees)")
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train_sc, y_train)
    y_pred2  = rf.predict(X_test_sc)

    mae2     = mean_absolute_error(y_test, y_pred2)
    rmse2    = np.sqrt(mean_squared_error(y_test, y_pred2))
    r2_2     = r2_score(y_test, y_pred2)
    adj_r2_2 = 1 - (1 - r2_2) * (n_test - 1) / (n_test - p1 - 1)
    nrmse2   = rmse2 / (y_test.max() - y_test.min())
    res2     = y_test.values - y_pred2

    print(f"  Model 2 (Random Forest) training complete")
    print(f"      Trees fitted        : 100")
    print(f"      Features used       : {rf.n_features_in_}")
    print(f"\n  Model 2 performance on test set ({n_test} records):")
    print(f"      MAE      = {mae2:.4f}")
    print(f"      RMSE     = {rmse2:.4f}")
    print(f"      NRMSE    = {nrmse2:.4f}")
    print(f"      R²       = {r2_2:.4f}  ({r2_2*100:.1f}% variance explained)")
    print(f"      Adj. R²  = {adj_r2_2:.4f}")

    # print(f"\n  Top 5 feature importances (Random Forest):")
    importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    # for feat, imp in importances.head(5).items():
    #     print(f"  {feat:<40} {imp:.4f}")
    # print()

except Exception as e:
    print(f"\n  ERROR during Model 2 training/evaluation: {e}")
    raise


# STEP 11 — Baseline model and model comparison

print("=" * 65)
print("STEP 11: Baseline model and full model comparison")
print("=" * 65)

try:
    y_base = np.full_like(y_test, y_train.mean(), dtype=float)
    mae_b  = mean_absolute_error(y_test, y_base)
    rmse_b = np.sqrt(mean_squared_error(y_test, y_base))
    r2_b   = r2_score(y_test, y_base)

    print(f"  Baseline model: predict mean AEER = {y_train.mean():.4f} for every record")
    print(f"\n  {'Metric':<12} {'Baseline':>12} {'OLS (M1)':>12} {'RandomForest (M2)':>18}")
    print(f"  {'─'*58}")
    print(f"  {'MAE':<12} {mae_b:>12.4f} {mae1:>12.4f} {mae2:>18.4f}")
    print(f"  {'RMSE':<12} {rmse_b:>12.4f} {rmse1:>12.4f} {rmse2:>18.4f}")
    print(f"  {'NRMSE':<12} {'—':>12} {nrmse1:>12.4f} {nrmse2:>18.4f}")
    print(f"  {'R²':<12} {r2_b:>12.4f} {r2_1:>12.4f} {r2_2:>18.4f}")
    print(f"  {'Adj. R²':<12} {'—':>12} {adj_r2_1:>12.4f} {adj_r2_2:>18.4f}")
    print(f"  {'─'*58}")

    if r2_1 <= r2_b:
        print("  WARNING: Model 1 (OLS) does not outperform the baseline — investigate features.")
    else:
        print(f"  Model 1 outperforms baseline (R² {r2_b:.4f} {r2_1:.4f})")
    if r2_2 <= r2_1:
        print("  WARNING: Model 2 (RF) did not improve over Model 1 — check for data issues.")
    else:
        print(f"  Model 2 outperforms Model 1 (R² {r2_1:.4f} {r2_2:.4f})")
    print()

except Exception as e:
    print(f"\n  ERROR during model comparison: {e}")
    raise


# STEP 12 — Statistical hypothesis test (Paired t-test)

print("=" * 65)
print("STEP 12: Statistical significance test — Paired t-test")
print("H₀: No difference in |residuals| OLS vs Random Forest")
print("H₁: Significant difference exists")
print("=" * 65)

try:
    abs_res1 = np.abs(res1)
    abs_res2 = np.abs(res2)

    if len(abs_res1) != len(abs_res2):
        raise ValueError(
            f"Residual arrays have different lengths: "
            f"{len(abs_res1)} vs {len(abs_res2)}. Cannot perform paired test."
        )

    t_stat, p_val = stats.ttest_rel(abs_res1, abs_res2)

    print(f"  Paired t-test results:")
    print(f"      Mean |residual| OLS : {abs_res1.mean():.4f}")
    print(f"      Mean |residual| RF  : {abs_res2.mean():.4f}")
    print(f"      t-statistic         : {t_stat:.4f}")
    print(f"      p-value             : {p_val:.6f}")
    print(f"      Significance level  : α = 0.05")

    if p_val < 0.05:
        winner = "Random Forest" if t_stat > 0 else "OLS"
        print(f"\n  REJECT H₀: Significant difference detected (p < 0.05)")
        print(f"      {winner} has significantly smaller errors")
    else:
        print(f"\n  FAIL TO REJECT H₀: No significant difference found (p ≥ 0.05)")

    print()

except ValueError as e:
    print(f"\n  VALUE ERROR in t-test: {e}")
    t_stat, p_val = 0.0, 1.0
except Exception as e:
    print(f"\n  ERROR in t-test: {e}")
    t_stat, p_val = 0.0, 1.0


# STEP 13 — Generate residual and diagnostic plots

# print("=" * 65)
# print("STEP 13: Generating residual diagnostic plots")
# print("=" * 65)

# Plot 5: Residuals vs fitted + Q-Q plot
try:
    # print(" Generating Plot 5: Residuals vs Fitted and Q-Q plot...")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].scatter(y_pred1, res1, alpha=0.4, s=15, color=TEAL)
    axes[0].axhline(0, color=CORAL, linestyle='--', linewidth=1.5)
    axes[0].set_xlabel('Fitted Values (AEER)', color=NAVY)
    axes[0].set_ylabel('Residuals', color=NAVY)
    axes[0].set_title('Model 1 OLS: Residuals vs Fitted',
                      fontsize=12, fontweight='bold', color=NAVY)

    stats.probplot(res1, dist='norm', plot=axes[1])
    axes[1].set_title('Model 1 OLS: Q-Q Plot of Residuals',
                      fontsize=12, fontweight='bold', color=NAVY)
    axes[1].get_lines()[0].set(color=TEAL, markersize=3, alpha=0.5)
    axes[1].get_lines()[1].set(color=CORAL, linewidth=2)

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plot5_residuals.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plot5_residuals.png saved")

except Exception as e:
    print(f"  ERROR generating Plot 5: {e}")
    plt.close('all')

# Plot 6: Standardised OLS coefficients
try:
    # print(" Generating Plot 6: OLS standardised coefficients...")
    coef_df = pd.DataFrame({
        'Feature':     feature_cols,
        'Coefficient': lr1.coef_,
    }).sort_values('Coefficient', key=abs, ascending=True)

    colors_bar = [CORAL if c < 0 else TEAL for c in coef_df['Coefficient']]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(coef_df['Feature'], coef_df['Coefficient'],
            color=colors_bar, edgecolor='white')
    ax.axvline(0, color=NAVY, linewidth=1)
    ax.set_title('Model 1 OLS: Standardised Coefficients',
                 fontsize=13, fontweight='bold', color=NAVY)
    ax.set_xlabel('Coefficient (standardised features)', color=NAVY)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plot6_coefficients.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plot6_coefficients.png saved")

except Exception as e:
    print(f"  ERROR generating Plot 6: {e}")
    plt.close('all')

# Plot 7: Model comparison bar chart
try:
    # print(" Generating Plot 7: Model performance comparison bar chart...")
    fig, ax = plt.subplots(figsize=(8, 4))
    metrics_labels = ['MAE', 'RMSE', 'R²']
    baseline_vals  = [mae_b,  rmse_b,  max(r2_b, 0)]
    model1_vals    = [mae1,   rmse1,   r2_1]
    model2_vals    = [mae2,   rmse2,   r2_2]
    x = np.arange(len(metrics_labels))
    w = 0.26
    ax.bar(x - w, baseline_vals, w, label='Baseline',                color='#BBBBBB', edgecolor='white')
    ax.bar(x,     model1_vals,   w, label='Model 1 (OLS)',           color=TEAL,      edgecolor='white')
    ax.bar(x + w, model2_vals,   w, label='Model 2 (Random Forest)', color=MINT,      edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_labels, fontsize=12)
    ax.set_title('Model Performance Comparison',
                 fontsize=13, fontweight='bold', color=NAVY)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/data_analytics/plot7_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    # print("  plot7_comparison.png saved")

except Exception as e:
    print(f"  ERROR generating Plot 7: {e}")
    plt.close('all')

# print(f"\n  All diagnostic plots generated.\n")


# STEP 14 — Save results to JSON

print("=" * 65)
print("STEP 14: Saving results summary to results.json")
print("=" * 65)

try:
    results = {
        'n_total':          n_total,
        'n_approved':       n_approved,
        'n_train':          int(X_train.shape[0]),
        'n_test':           int(X_test.shape[0]),
        'n_features':       len(feature_cols),
        'feature_names':    feature_cols,
        'cd_brands_matched': int(df['CD_Avg_Star'].notna().sum()) if 'CD_Avg_Star' in df.columns else 0,
        'aeer_mean':        round(float(df['Rated_AEER'].mean()), 3),
        'aeer_std':         round(float(df['Rated_AEER'].std()),  3),
        'baseline_mae':     round(mae_b,  4),
        'baseline_rmse':    round(rmse_b, 4),
        'ols_mae':          round(mae1,     4),
        'ols_rmse':         round(rmse1,    4),
        'ols_r2':           round(r2_1,     4),
        'ols_adj_r2':       round(adj_r2_1, 4),
        'ols_nrmse':        round(nrmse1,   4),
        'ols_intercept':    round(float(lr1.intercept_), 6),
        'matrix_inv_intercept': round(float(beta_inv[0]), 6),
        'rf_mae':           round(mae2,     4),
        'rf_rmse':          round(rmse2,    4),
        'rf_r2':            round(r2_2,     4),
        'rf_adj_r2':        round(adj_r2_2, 4),
        'rf_nrmse':         round(nrmse2,   4),
        'ttest_statistic':  round(t_stat,   4),
        'ttest_pvalue':     round(p_val,    6),
        'shapiro_w':        round(sw_stat,  4),
        'shapiro_p':        round(sw_p,     4),
        'max_vif_feature':  max_vif_feat,
        'max_vif_value':    max_vif_val,
    }

    with open('/content/drive/MyDrive/data_analytics/results.json', 'w') as f:
        json.dump(results, f, indent=2)

    print("  results.json saved successfully")
    print(f"\n  Key results summary:")
    print(f"      OLS  R²             : {r2_1:.4f}  ({r2_1*100:.1f}% variance explained)")
    print(f"      RF   R²             : {r2_2:.4f}  ({r2_2*100:.1f}% variance explained)")
    print(f"      Paired t-test       : t={t_stat:.4f},  p={p_val:.6f}")
    print(f"      Shapiro-Wilk (AEER) : W={sw_stat:.4f},  p={sw_p:.4f}")
    print(f"      Max VIF             : {max_vif_val} ({max_vif_feat})")
    print()

except (IOError, OSError) as e:
    print(f"\n  FILE WRITE ERROR: {e}")
    print("     Check that you have write permissions to the working directory.")
except Exception as e:
    print(f"\n  UNEXPECTED ERROR saving results: {e}")


# ======================== PIPELINE COMPLETE =======================

print("=" * 65)
print()
print("Output files generated:")
print("results.json")


  All libraries loaded successfully.

  Dataset loaded successfully.
     Total rows    : 5671
     Total columns : 167

  Missing values detected in 138 columns.

STEP 3: Heterogeneous data integration

  Dryer brand merge results:
      AC rows matched to dryer brand data : 1595  (28.1%)
      AC rows unmatched (no dryer data)   : 4076 (71.9%) will be imputed
      Row count preserved                 : 5671
  New columns added: CD_Avg_Energy_kWh, CD_Avg_Star, CD_Avg_Cap_kg

STEP 4: Filtering, encoding, and feature engineering
        (Pre-split preprocessing steps)
  Filtered to Approved status: 5671 5668 records (3 removed)
  No rows dropped for missing AEER

  Pre-split preprocessing complete. Records remaining: 5668

STEP 5: Exploratory Data Analysis — generating visualisations
  plot2_boxplots.png saved

  EDA complete — all visualisations generated.

STEP 6: Feature selection and train/test split (80:20)
  Dryer brand features included: ['CD_Avg_Energy_kWh', 'CD_Avg_Star', 'CD_